In [ ]:
# Imports
import sys
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import signal as scipy_signal
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from src.preprocessing import ECGPreprocessor
from src.signal_separation import SignalSeparator

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
print('Imports successful.')

In [ ]:
# Configuration
SAMPLING_RATE = 500
DURATION_SEC  = 10
N_COMPONENTS  = 2
RESULTS_DIR   = Path('../results/plots')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = SAMPLING_RATE * DURATION_SEC
t = np.linspace(0, DURATION_SEC, N_SAMPLES)
print(f'{N_SAMPLES} samples @ {SAMPLING_RATE} Hz ({DURATION_SEC} s)')

In [ ]:
# Synthetic ECG template (replace with wfdb.rdrecord() for real data)
def make_ecg_template(t, heart_rate_bpm, amplitude=1.0, noise_std=0.05):
    rr = 60.0 / heart_rate_bpm
    ecg = np.zeros_like(t)
    for bt in np.arange(0, t[-1], rr):
        ecg += amplitude * np.exp(-((t - bt) ** 2) / (2 * 0.02 ** 2))
    ecg += noise_std * np.random.randn(len(t))
    return ecg

np.random.seed(42)
maternal_pure = make_ecg_template(t, heart_rate_bpm=75,  amplitude=1.0, noise_std=0.04)
fetal_pure    = make_ecg_template(t, heart_rate_bpm=145, amplitude=0.4, noise_std=0.02)

# Mixing matrix: each channel = weighted sum of both sources
A = np.array([[0.8, 0.4],
              [0.3, 0.9]])
sources = np.column_stack([maternal_pure, fetal_pure])
mixed   = sources @ A.T

print(f'Sources: {sources.shape}  Mixed: {mixed.shape}')
print(f'Mixing matrix A:\n{A}')

In [ ]:
# Plot mixed signals
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(t, mixed[:, i], linewidth=0.7, color=f'C{i}')
    ax.set_ylabel(f'CH {i+1} (mV)'); ax.set_title(f'Mixed ECG — Channel {i+1}')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb02_01_mixed_signals.png', bbox_inches='tight')
plt.show()

In [ ]:
# Bandpass filter + z-score normalisation
preprocessor = ECGPreprocessor(sampling_rate=SAMPLING_RATE, lowcut=0.5, highcut=40)
filtered = preprocessor.filter_signal(mixed)
filtered = preprocessor.normalize_signal(filtered, method='zscore')
print(f'Filtered: {filtered.shape}  means={filtered.mean(axis=0).round(4)}  stds={filtered.std(axis=0).round(4)}')

In [ ]:
# FastICA separation
separator = SignalSeparator(n_components=N_COMPONENTS, random_state=42)
components = separator.fit_transform(filtered)

print(f'Components: {components.shape}')
print(f'Mixing matrix:\n{separator.get_mixing_matrix().round(3)}')
print(f'Unmixing matrix:\n{separator.get_unmixing_matrix().round(3)}')

In [ ]:
# Identify components by dominant frequency (FFT)
# Maternal: 60–100 bpm (1.0–1.67 Hz)  |  Fetal: 120–160 bpm (2.0–2.67 Hz)

def dominant_frequency(sig, fs):
    freqs = np.fft.rfftfreq(len(sig), d=1/fs)
    power = np.abs(np.fft.rfft(sig)) ** 2
    mask = (freqs >= 0.5) & (freqs <= 4.0)
    return freqs[mask][np.argmax(power[mask])]

labels, dom_bpm = {}, {}
for i in range(N_COMPONENTS):
    dom_hz = dominant_frequency(components[:, i], SAMPLING_RATE)
    bpm = dom_hz * 60
    dom_bpm[i] = bpm
    if 60 <= bpm <= 100:
        labels[i] = 'Maternal'
    elif 100 < bpm <= 200:
        labels[i] = 'Fetal'
    else:
        labels[i] = f'Unknown ({bpm:.0f} bpm)'
    print(f'  IC {i}: {dom_hz:.2f} Hz -> {bpm:.0f} bpm -> [{labels[i]}]')

In [ ]:
# Plot separated components
colors = {'Maternal': 'steelblue', 'Fetal': 'tomato'}
fig, axes = plt.subplots(N_COMPONENTS, 1, figsize=(14, 5), sharex=True)
for i, ax in enumerate(axes):
    label = labels[i]
    ax.plot(t, components[:, i], linewidth=0.7, color=colors.get(label, 'grey'))
    ax.set_ylabel('Amplitude'); ax.set_title(f'IC {i+1}: {label} ({dom_bpm[i]:.0f} bpm)')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb02_02_separated_components.png', bbox_inches='tight')
plt.show()

In [ ]:
# Reconstruction quality
reconstructed = components @ separator.get_mixing_matrix().T
quality = separator.estimate_quality(filtered, reconstructed)
print(f'NMSE: {quality["nmse"]:.5f}  Correlation: {quality["correlation"]:.4f}')

# Cross-correlation (ideal = identity)
cross_corr = np.corrcoef(components.T)
print(f'Cross-correlation:\n{np.round(cross_corr, 4)}')

In [ ]:
# Power spectra of separated components
fig, axes = plt.subplots(1, N_COMPONENTS, figsize=(14, 4))
for i, ax in enumerate(axes):
    freqs, psd = scipy_signal.welch(components[:, i], fs=SAMPLING_RATE, nperseg=512)
    mask = freqs <= 5.0
    ax.semilogy(freqs[mask], psd[mask], color=colors.get(labels[i], 'grey'))
    ax.axvspan(1.0, 1.67, alpha=0.15, color='steelblue', label='Maternal range')
    ax.axvspan(2.0, 2.67, alpha=0.15, color='tomato',    label='Fetal range')
    ax.set_title(f'PSD — IC {i+1}: {labels[i]}'); ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb02_03_power_spectra.png', bbox_inches='tight')
plt.show()

In [ ]:
# Save separated signals to CSV + SQLite database
data_processed = Path('../data/processed')
data_processed.mkdir(parents=True, exist_ok=True)

col_names = [labels[i].lower() + '_ecg' for i in range(N_COMPONENTS)]
df_components = pd.DataFrame(components, columns=col_names)
df_components['time_s'] = t

# CSV output (kept for compatibility)
out_csv = data_processed / 'separated_components.csv'
df_components.to_csv(out_csv, index=False)

# Database output (direct DB workflow)
db_path = data_processed / 'neuro_genomic.db'
with sqlite3.connect(db_path) as conn:
    df_components.to_sql('separated_components', conn, if_exists='replace', index=False)

print(f'Saved CSV: {out_csv}')
print(f'Saved DB table: separated_components in {db_path}')
print(df_components.head())